In [1]:
import pandas as pd
import random
from collections import defaultdict, Counter

In [ ]:
POPULASI = 1500
VIOLATION_COST = 100
ITERATION = 10000
MUTATION_PROB = 0.8
CROSSOVER_PROB = 0.7 
TOURNAMENT_SIZE = 20
SLOT_PER_KELAS = 36
JUMLAH_KELAS = 27

In [3]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')
wali_kelas_df = pd.read_csv('../dataset/wali_kelas.csv')

In [4]:
hariId = {
"Senin": 1,
"Selasa": 2,
"Rabu": 3,
"Kamis": 4,
"Jumat": 5,
}

slotPerHari = slot_df['hari'].value_counts().to_dict()
slotPerHari = {hariId[k]:v for k,v in slotPerHari.items()}

guruPengajar = dict(
    zip(guru_df['guru_id'], guru_df['nama_guru'])
)

kelasDanTingkatan = defaultdict(list)

for row in kelas_df.itertuples():
    kelasDanTingkatan[row.kelas_id].append({
        'tingkatan': row.tingkatan,
        'nama_kelas': row.nama_kelas
    })

kelasDanTingkatan = dict(kelasDanTingkatan)

namaMapelDanId = dict(
    zip(mapel_df['mapel_id'], mapel_df['nama_mapel'])
)

jamPerMingguMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['jam_per_minggu'])
)

mgmpMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['MGMP'])
)

mgmpMapel = {mapel_id: hariId[hari] for mapel_id, hari in mgmpMapel.items()}

batasSiang = {1:5,2:5,3:4,4:5,5:4}
batasMGMP = {1:2,2:2,3:2,4:2,5:1}

durasiGuruMengajar = defaultdict(list)

for row in relasi_guru_mapel_df.itertuples():
    durasiGuruMengajar[row.guru_id].append({
        "mapel_id": row.mapel_id,
        "tingkatan": row.tingkatan,
        "durasi": row.durasi
    })

durasiGuruMengajar = dict(durasiGuruMengajar)

waliKelas = dict(
    zip(wali_kelas_df['guru_id'], wali_kelas_df['kelas_id'])
)

kelasIndex = {}

slotRange = 0

for i in range(1, JUMLAH_KELAS+1):

    end = slotRange + SLOT_PER_KELAS

    kelasIndex[i] = (slotRange, end)

    slotRange = end

In [5]:
def blokDistribusi(jam):

    if jam == 2:
        return [2]

    if jam == 3:
        return [3]

    if jam == 4:
        return [2,2]

    if jam == 5:
        return [2,3]

    return [jam]

In [6]:
def ambilGuruValid(mapel_id, tingkatan):

    listGuru = []

    for guru_id, relasiList in durasiGuruMengajar.items():

        for relasi in relasiList:

            if relasi["mapel_id"] == mapel_id and relasi["tingkatan"] == tingkatan:

                listGuru.append(guru_id)

    return listGuru

In [7]:
def perluasBlok(mapel_id, guru_id, durasi):

    return [(mapel_id, guru_id)] * durasi

In [8]:
def generatePerKelas(tingkatan):

    pilihan = []

    for mapel_id, jam in jamPerMingguMapel.items():

        blok = blokDistribusi(jam)

        guruValid = ambilGuruValid(mapel_id, tingkatan)

        if not guruValid:
            continue

        guru = random.choice(guruValid)

        for durasi in blok:

            pilihan.extend(perluasBlok(mapel_id, guru, durasi))

    random.shuffle(pilihan)

    return pilihan

In [9]:
def individuConstruct(kelas_id):

    tingkatan = kelasDanTingkatan[kelas_id][0]["tingkatan"]

    return generatePerKelas(tingkatan)

In [10]:
def individuTrigger():

    individu = []

    for kelas_id in sorted(kelasDanTingkatan.keys()):

        individu.extend(individuConstruct(kelas_id))

    return individu

In [11]:
def populasiConstruct(POPULASI):

    return [individuTrigger() for _ in range(POPULASI)]

populasiOptimasi = populasiConstruct(POPULASI)

In [12]:
slotAwalHari = {}

index = 0

for hari, jumlah in slotPerHari.items():

    slotAwalHari[hari] = index

    index += jumlah


slotKeHari = {}

index = 0

for hari, jumlah in slotPerHari.items():

    for _ in range(jumlah):

        slotKeHari[index] = hari

        index += 1

In [13]:
def guruBentrok(individu):

    pelanggaran = 0

    for slot in range(SLOT_PER_KELAS):

        guruMengajar = []

        for kelas in range(JUMLAH_KELAS):

            index = kelas * SLOT_PER_KELAS + slot

            guruMengajar.append(individu[index][1])

        if len(guruMengajar) != len(set(guruMengajar)):

            pelanggaran += 1

    return pelanggaran

In [14]:
def durasiGuru(individu):

    loadGuru = Counter(guru for _,guru in individu)

    pelanggaran = 0

    for guru,count in loadGuru.items():

        if count > 40:

            pelanggaran += count - 40

    return pelanggaran

In [15]:
def distribusiMapel(individu):

    pelanggaran = 0

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS
        end = start + SLOT_PER_KELAS

        kelasSlot = individu[start:end]

        mapelCount = {}

        for mapel,guru in kelasSlot:

            mapelCount[mapel] = mapelCount.get(mapel,0) + 1

        for mapel_id, jam in jamPerMingguMapel.items():

            if mapelCount.get(mapel_id,0) != jam:
                pelanggaran += 1

    return pelanggaran

In [16]:
def mapelSiang(individu):

    pelanggaran = 0

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS

        for slot in range(SLOT_PER_KELAS):

            index = start + slot

            mapel_id = individu[index][0]

            hari = slotKeHari[slot]

            batas = batasSiang[hari]

            slotHari = slot - slotAwalHari[hari]

            if slotHari >= batas:

                pelanggaran += 1

    return pelanggaran

In [17]:
def waktuMGMP(individu):

    pelanggaran = 0

    for kelas in range(JUMLAH_KELAS):

        start = kelas * SLOT_PER_KELAS

        for slot in range(SLOT_PER_KELAS):

            index = start + slot

            mapel_id = individu[index][0]

            hari = slotKeHari[slot]

            if mapel_id in mgmpMapel:

                if hari == mgmpMapel[mapel_id]:

                    pelanggaran += 1

    return pelanggaran

In [18]:
def cekWaliKelas(individu):

    pelanggaran = 0

    for guru_id,kelas_id in waliKelas.items():

        start = (kelas_id-1) * SLOT_PER_KELAS
        end = start + SLOT_PER_KELAS

        kelasSlot = individu[start:end]

        mengajar = False

        for mapel,guru in kelasSlot:

            if guru == guru_id:
                mengajar = True
                break

        if not mengajar:
            pelanggaran += 1

    return pelanggaran

In [19]:
def evaluasiIndividu(individu):

    pelanggaran = 0

    pelanggaran += guruBentrok(individu)

    pelanggaran += distribusiMapel(individu)

    pelanggaran += mapelSiang(individu)

    pelanggaran += durasiGuru(individu)

    pelanggaran += waktuMGMP(individu)

    pelanggaran += cekWaliKelas(individu)

    return pelanggaran * VIOLATION_COST

In [20]:
def slotBermasalah(individu):

    masalah = set()

    slotPerkelas = 36
    jumlahKelas = len(kelasIndex)

    for slot in range(slotPerkelas):

        guruIndex = {}

        index = slot

        for _ in range(jumlahKelas):

            _, guru = individu[index]

            if guru in guruIndex:

                masalah.add(index)
                masalah.add(guruIndex[guru])

            else:
                guruIndex[guru] = index

            index += slotPerkelas

    return list(masalah)

In [21]:
def kelasBermasalah(individu):

    konflik = {}

    bermasalah = set(slotBermasalah(individu))

    for kelas, (start, end) in kelasIndex.items():

        konflik[kelas] = sum(1 for i in range(start,end) if i in bermasalah)

    return max(konflik, key=konflik.get)

In [22]:
def clusterMapel(slot):

    kelompok = defaultdict(list)

    for mapel, guru in slot:
        kelompok[mapel].append((mapel, guru))

    hasil = []

    keys = list(kelompok.keys())
    random.shuffle(keys)

    for mapel in keys:
        hasil.extend(kelompok[mapel])

    return hasil

In [23]:
def mutasi(individu, MUTATION_PROB):

    child = individu.copy()

    if random.random() > MUTATION_PROB:
        return child

    # pilih kelas bermasalah jika ada
    masalah = slotBermasalah(child)

    if masalah:
        kelas = kelasBermasalah(child)
    else:
        kelas = random.choice(list(kelasIndex.keys()))

    start, end = kelasIndex[kelas]

    slotKelas = child[start:end]

    slotKelas = clusterMapel(slotKelas)

    random.shuffle(slotKelas)

    child[start:end] = slotKelas

    return child

def mutasiTargeted(individu):

    child = individu.copy()

    masalah = slotBermasalah(child)

    if not masalah:
        return child

    i = random.choice(masalah)

    slotPerkelas = SLOT_PER_KELAS

    kelas = i // slotPerkelas

    start = kelas * slotPerkelas
    end = start + slotPerkelas

    j = random.randint(start, end-1)

    child[i], child[j] = child[j], child[i]

    return child

def mutasiRepair(individu):

    child = individu.copy()

    masalah = slotBermasalah(child)

    if not masalah:
        return child

    # pilih slot yang bermasalah
    i = random.choice(masalah)

    mapel_i, guru_i = child[i]

    # cari slot lain untuk ditukar
    percobaan = 100

    for _ in range(percobaan):

        j = random.randint(0, len(child)-1)

        if i == j:
            continue

        mapel_j, guru_j = child[j]

        # hindari swap dengan guru yang sama
        if guru_i != guru_j:

            child[i], child[j] = child[j], child[i]

            # cek apakah konflik berkurang
            if len(slotBermasalah(child)) <= len(masalah):
                return child
            else:
                # rollback jika lebih buruk
                child[i], child[j] = child[j], child[i]

    return child

In [24]:
def crossover(parent1, parent2):

    child = parent1.copy()

    jumlah = random.randint(2,4)

    kelasDipilih = random.sample(list(kelasIndex.keys()), jumlah)

    for kelas in kelasDipilih:

        start, end = kelasIndex[kelas]

        child[start:end] = parent2[start:end]

    return child

def crossoverTargeted(parent1, parent2):

    child = parent1.copy()

    kelasUtama = kelasBermasalah(parent1)

    start, end = kelasIndex[kelasUtama]

    child[start:end] = parent2[start:end]

    # tambah 1–2 kelas random untuk variasi
    tambahan = random.sample(list(kelasIndex.keys()), random.randint(1,2))

    for kelas in tambahan:

        start, end = kelasIndex[kelas]

        child[start:end] = parent2[start:end]

    return child

In [25]:
def turnamen(populasi, fitnessPop):

    kandidat = random.sample(range(len(populasi)), TOURNAMENT_SIZE)

    terbaik = min(kandidat, key=lambda i: fitnessPop[i])

    return populasi[terbaik]

In [26]:
def geneticAlgorithm(populasiAwal):

    populasi = populasiAwal

    bestIndividu = None
    bestFitness = float("inf")

    for gen in range(ITERATION):

        fitnessPop = []

        for individu in populasi:

            fitness = evaluasiIndividu(individu)

            fitnessPop.append(fitness)

            if fitness < bestFitness:

                bestFitness = fitness
                bestIndividu = individu

        print("Generasi:", gen, "Best Fitness:", bestFitness)

        elitIndex = sorted(range(len(fitnessPop)), key=lambda i: fitnessPop[i])

        elit = [populasi[i] for i in elitIndex[:2]]

        populasiBaru = elit.copy()

        while len(populasiBaru) < POPULASI:

            parent1 = turnamen(populasi, fitnessPop)

            parent2 = turnamen(populasi, fitnessPop)

            # child = crossover(parent1,parent2)

            # child = mutasi(child,MUTATION_PROB)

            if random.random() < CROSSOVER_PROB:
                if random.random() < 0.5:
                    child = crossover(parent1, parent2)
                else:
                    child = crossoverTargeted(parent1, parent2)
            else:
                child = parent1.copy()

            if random.random() < MUTATION_PROB:
                child = mutasi(child, MUTATION_PROB)

            if random.random() < 0.5:
                child = mutasiTargeted(child)

            if random.random() < 0.3:
                child = mutasiRepair(child)

            populasiBaru.append(child)

        populasi = populasiBaru

    return bestIndividu,bestFitness

In [27]:
hasil = geneticAlgorithm(populasiOptimasi)

Generasi: 0 Best Fitness: 56300
Generasi: 1 Best Fitness: 55900
Generasi: 2 Best Fitness: 54800
Generasi: 3 Best Fitness: 54300
Generasi: 4 Best Fitness: 53600
Generasi: 5 Best Fitness: 52800
Generasi: 6 Best Fitness: 51900
Generasi: 7 Best Fitness: 51500
Generasi: 8 Best Fitness: 50900
Generasi: 9 Best Fitness: 50100
Generasi: 10 Best Fitness: 49800
Generasi: 11 Best Fitness: 49300
Generasi: 12 Best Fitness: 49200
Generasi: 13 Best Fitness: 48600
Generasi: 14 Best Fitness: 48400
Generasi: 15 Best Fitness: 48200
Generasi: 16 Best Fitness: 48100
Generasi: 17 Best Fitness: 47800
Generasi: 18 Best Fitness: 47600
Generasi: 19 Best Fitness: 47400
Generasi: 20 Best Fitness: 47200
Generasi: 21 Best Fitness: 47100
Generasi: 22 Best Fitness: 47000
Generasi: 23 Best Fitness: 46900
Generasi: 24 Best Fitness: 46800
Generasi: 25 Best Fitness: 46700
Generasi: 26 Best Fitness: 46700
Generasi: 27 Best Fitness: 46500
Generasi: 28 Best Fitness: 46400
Generasi: 29 Best Fitness: 46300
Generasi: 30 Best Fi